# Tarea 2.2 — Herramienta de Exploración y Preparación de Datos

**Araceli Castillo** · MAI 540: Machine Learning · Prof. Kevin A. Garcia Gallardo
Atlantis University · Septiembre de 2026

Proyecto: **Mantenimiento predictivo** — predecir `failure_next_7_days` (1 = la máquina falla en los próximos 7 días).
Repositorio: https://github.com/ACAS0802/mantenimiento-predictivo-mai540

---

## El orden de las transformaciones

Este cuaderno está organizado en el orden exacto que evita la fuga de información. No es un orden estético: cada paso está donde está por una razón que se explica en su sección.

| # | Paso | ¿Por qué va aquí? |
|---|---|---|
| 1 | Cargar datos crudos | Todavía no se transforma nada |
| 2 | Diagnóstico | Solo se mira, no se modifica |
| 3 | Limpieza sin estadísticas | No aprende parámetros, así que puede ir antes del split — y los duplicados **deben** quitarse antes, o una fila queda en train y su copia en test |
| 4 | **`train_test_split`** | **La frontera.** Nada de lo que sigue puede mirar el conjunto de prueba |
| 5 | Imputación + escalamiento + codificación | Dentro de un `Pipeline` que hace `fit` solo con train |
| 6 | Selección de características | Decidida con validación cruzada **sobre train**. Es el paso donde la fuga sí se materializa (ver sección 8) |
| 7 | Evaluación en test | Una sola vez, al final |

## 0. Preparación del entorno

En Google Colab, ejecuta esta celda primero. Clona el repositorio para tener el dataset y el módulo `src/preprocesamiento.py`.

In [1]:
# Funciona igual en Colab y en local, y sin importar desde qué carpeta se abra.
import os, sys

def raiz_del_proyecto():
    # 1) ya estamos en la raíz
    if os.path.exists("data/datos.csv"):
        return os.getcwd()
    # 2) el cuaderno se abrió desde notebooks/
    if os.path.exists("../data/datos.csv"):
        return os.path.abspath("..")
    # 3) Colab: clonar el repositorio
    if not os.path.exists("mantenimiento-predictivo-mai540"):
        os.system("git clone -q https://github.com/ACAS0802/mantenimiento-predictivo-mai540.git")
    return os.path.abspath("mantenimiento-predictivo-mai540")

os.chdir(raiz_del_proyecto())
sys.path.insert(0, "src")
os.makedirs("reportes", exist_ok=True)

assert os.path.exists("data/datos.csv"), "No se encontró data/datos.csv"
print("Raíz del proyecto:", os.getcwd())
print("Dataset encontrado. Todo listo.")

Raíz del proyecto: /sessions/rcw-01g3maktyqzkfaczdfedvxkx/t22
Dataset encontrado. Todo listo.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 42
TEST_SIZE = 0.25
OBJETIVO = "failure_next_7_days"

NUMERICAS = ["operating_hours", "temperature_c", "vibration_mm_s", "pressure_bar",
             "days_since_maintenance", "error_count_30d", "energy_kw"]
CATEGORICAS = ["machine_type", "shift", "environment"]

print("Librerías cargadas.")

Librerías cargadas.


## 1. Cargar los datos crudos

In [3]:
df = pd.read_csv("data/datos.csv")
print(f"Filas: {len(df):,}   Columnas: {df.shape[1]}")
df.head()

Filas: 5,000   Columnas: 11


,machine_type,operating_hours,temperature_c,vibration_mm_s,pressure_bar,days_since_maintenance,shift,environment,error_count_30d,energy_kw,failure_next_7_days
0,A,3743,84.93,5.64,7.06,76,noche,seco,1,39.02,0
1,A,6478,58.50,4.84,6.96,69,tarde,humedo,4,53.89,0
2,B,5068,53.86,3.09,5.84,138,manana,humedo,2,41.66,0
3,A,4699,67.94,4.92,7.40,122,tarde,seco,3,34.26,0
4,B,6497,78.92,4.21,5.85,202,tarde,humedo,1,36.76,0


## 2. Diagnóstico del estado actual

Antes de tocar nada, hay que saber qué hay. Esta sección solo observa.

In [4]:
diagnostico = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "faltantes": df.isna().sum(),
    "% faltante": (df.isna().mean() * 100).round(2),
    "valores únicos": df.nunique(),
})
print("--- Estado de cada variable ---")
print(diagnostico.to_string())

print(f"\n--- Variable objetivo ---")
print(df[OBJETIVO].value_counts().to_string())
print(f"Clase positiva: {df[OBJETIVO].mean()*100:.2f}%  ->  problema DESBALANCEADO")
print("Por eso accuracy no sirve sola: un modelo que diga siempre 0 acierta el 91% y no detecta ninguna falla.")

print(f"\n--- Duplicados ---")
predictoras = [c for c in df.columns if c != OBJETIVO]
print(f"Filas 100% duplicadas: {df.duplicated().sum()}")
print(f"Duplicadas solo en predictoras: {df.duplicated(subset=predictoras).sum()}")

--- Estado de cada variable ---
                           tipo  faltantes  % faltante  valores únicos
machine_type             object          0        0.00               3
operating_hours           int64          0        0.00            3666
temperature_c           float64          0        0.00            2524
vibration_mm_s          float64        110        2.20             536
pressure_bar            float64        168        3.36             381
days_since_maintenance    int64          0        0.00             258
shift                    object          0        0.00               3
environment              object        113        2.26               3
error_count_30d           int64          0        0.00              11
energy_kw               float64          0        0.00            2351
failure_next_7_days       int64          0        0.00               2

--- Variable objetivo ---
failure_next_7_days
0    4556
1     444
Clase positiva: 8.88%  ->  problema DESBALANCEADO

### 2.1 ¿Los faltantes son aleatorios?

Esta pregunta decide la estrategia de imputación. Si el hecho de que un valor falte está relacionado con la variable objetivo, el faltante *en sí mismo* es información.

In [5]:
for col in ["vibration_mm_s", "pressure_bar", "environment"]:
    falta = df[col].isna()
    print(f"{col}  ({falta.sum()} faltantes, {falta.mean()*100:.2f}%)")
    print(f"   tasa de falla cuando FALTA    : {df.loc[falta, OBJETIVO].mean()*100:6.2f}%")
    print(f"   tasa de falla cuando NO falta : {df.loc[~falta, OBJETIVO].mean()*100:6.2f}%")
    print()

vibration_mm_s  (110 faltantes, 2.20%)
   tasa de falla cuando FALTA    :   4.55%
   tasa de falla cuando NO falta :   8.98%

pressure_bar  (168 faltantes, 3.36%)
   tasa de falla cuando FALTA    :  11.90%
   tasa de falla cuando NO falta :   8.77%

environment  (113 faltantes, 2.26%)
   tasa de falla cuando FALTA    :   8.85%
   tasa de falla cuando NO falta :   8.88%



**Lectura.** En `pressure_bar` la tasa de falla es más alta cuando el valor falta (11.90 % contra 8.77 %), y en `vibration_mm_s` es más baja (4.55 % contra 8.98 %). Es decir, **los faltantes no son completamente aleatorios**.

Eso hace razonable probar un indicador de faltante como tercera estrategia en la sección 5. Lo importante es que esa hipótesis se **prueba con datos**, no se asume.

## 3. Limpieza que NO depende de estadísticas

Este es el único trabajo que puede hacerse antes del split, porque no aprende ningún parámetro del conjunto: no calcula medias, medianas ni cuantiles.

Los duplicados exactos **deben** quitarse aquí, antes de partir. Si una fila repetida queda en train y su copia idéntica en test, el modelo ya vio la respuesta: eso es fuga, y ningún `Pipeline` la corrige después.

In [6]:
filas_antes = len(df)

# 3a. Duplicados exactos
duplicados = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# 3b. Reglas físicas del dominio (no son estadísticas: ninguna de estas magnitudes
#     puede ser negativa). Un valor imposible pasa a faltante y se imputa después.
imposibles = 0
for col in NUMERICAS:
    malos = int((df[col] < 0).sum())
    imposibles += malos
    if malos:
        df.loc[df[col] < 0, col] = np.nan

print(f"Filas antes ................. {filas_antes:,}")
print(f"Duplicados exactos quitados . {duplicados}")
print(f"Valores imposibles -> NaN ... {imposibles}")
print(f"Filas después ............... {len(df):,}")
print("\nEn este dataset no había ninguno de los dos. Se deja el código igual:")
print("la herramienta debe funcionar con datos que sí los tengan.")

Filas antes ................. 5,000
Duplicados exactos quitados . 0
Valores imposibles -> NaN ... 0
Filas después ............... 5,000

En este dataset no había ninguno de los dos. Se deja el código igual:
la herramienta debe funcionar con datos que sí los tengan.


## 4. La frontera: `train_test_split`

A partir de esta línea, el conjunto de prueba deja de existir hasta la sección 9.

Se usa `stratify=y` porque la clase positiva es solo el 8.88 %: sin estratificar, una partición desafortunada podría dejar muy pocas fallas en el test.

In [7]:
X = df[NUMERICAS + CATEGORICAS]
y = df[OBJETIVO].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEMILLA, stratify=y
)

print(f"train: {len(X_train):,} filas  ({y_train.mean()*100:.2f}% fallas)")
print(f"test : {len(X_test):,} filas  ({y_test.mean()*100:.2f}% fallas)")
print("\nA partir de aquí, X_test y y_test NO se tocan hasta la sección 9.")

train: 3,750 filas  (8.88% fallas)
test : 1,250 filas  (8.88% fallas)

A partir de aquí, X_test y y_test NO se tocan hasta la sección 9.


## 5. Datos faltantes: comparación de tres estrategias

La tarea pide comparar al menos dos. Comparo tres, y **toda la comparación se hace con validación cruzada sobre train**: si midiera en test, la decisión de qué imputador usar ya estaría contaminada.

| Estrategia | Idea |
|---|---|
| A — Mediana | Robusta a atípicos, no asume forma de la distribución |
| B — KNN (k=5) | Usa las filas más parecidas; más sofisticada y más cara |
| C — Mediana + indicador | Añade una columna binaria "este valor faltaba", por lo que vimos en 2.1 |

In [8]:
def arma_pipeline(imputador_num, numericas=NUMERICAS, categoricas=CATEGORICAS):
    return Pipeline([
        ("preprocesamiento", ColumnTransformer([
            ("num", Pipeline([("imp", imputador_num), ("esc", StandardScaler())]), numericas),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("cod", OneHotEncoder(handle_unknown="ignore"))]), categoricas),
        ])),
        ("modelo", LogisticRegression(max_iter=1000, random_state=SEMILLA,
                                      class_weight="balanced")),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
estrategias = {
    "A) Mediana":                      SimpleImputer(strategy="median"),
    "B) KNN (k=5)":                    KNNImputer(n_neighbors=5),
    "C) Mediana + indicador faltante": SimpleImputer(strategy="median", add_indicator=True),
}

print(f"{'Estrategia':36s}{'F1 clase 1':>12s}{'Recall':>10s}{'Precisión':>11s}")
print("-" * 69)
for nombre, imp in estrategias.items():
    p = arma_pipeline(imp)
    f1 = cross_val_score(p, X_train, y_train, cv=cv, scoring="f1").mean()
    rc = cross_val_score(p, X_train, y_train, cv=cv, scoring="recall").mean()
    pr = cross_val_score(p, X_train, y_train, cv=cv, scoring="precision").mean()
    print(f"{nombre:36s}{f1:12.4f}{rc:10.4f}{pr:11.4f}")

Estrategia                            F1 clase 1    Recall  Precisión
---------------------------------------------------------------------


A) Mediana                                0.2650    0.6578     0.1660


B) KNN (k=5)                              0.2647    0.6578     0.1659


C) Mediana + indicador faltante           0.2621    0.6487     0.1643


**Decisión: mediana (estrategia A).**

Y la justificación honesta no es "es la mejor", porque no lo es: **las tres son equivalentes** (F1 de 0.2650, 0.2647 y 0.2621 — diferencias en la tercera cifra decimal, muy por debajo de la variabilidad entre particiones).

Cuando tres opciones empatan, gana la más simple. La mediana:

- es robusta a valores atípicos, y en la sección 6 se demuestra que aquí los atípicos se conservan;
- no añade columnas al modelo, a diferencia de C, que crearía variables nuevas fuera de las diez autorizadas por `CONTEXT.md`;
- es mucho más barata que KNN, que tiene que calcular distancias entre filas.

El indicador de faltante era una hipótesis razonable después de lo que vimos en 2.1, y **los datos la descartaron**. Con solo 2–3 % de faltantes no hay suficiente señal para que aporte.

## 6. Valores atípicos: ¿ruido o señal?

La consigna es clara: no basta con eliminar un valor porque "se ve raro". Aquí la pregunta se responde con una medición — **¿las filas atípicas fallan más o menos que las normales?**

Los límites IQR se calculan **solo con train**. Calcularlos con todo el conjunto sería usar información del test para decidir qué filas borrar.

In [9]:
entrenamiento = X_train.copy()
entrenamiento[OBJETIVO] = y_train
tasa_global = entrenamiento[OBJETIVO].mean() * 100

print(f"Tasa de falla global en train: {tasa_global:.2f}%\n")
print(f"{'variable':26s}{'atípicos':>10s}{'falla en atíp.':>16s}{'falla normales':>16s}{'lift':>8s}")
print("-" * 76)

limites = {}
for col in NUMERICAS:
    q1, q3 = entrenamiento[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    limites[col] = (lo, hi)
    fuera = (entrenamiento[col] < lo) | (entrenamiento[col] > hi)
    if fuera.sum() == 0:
        print(f"{col:26s}{0:>10d}")
        continue
    f_out = entrenamiento.loc[fuera, OBJETIVO].mean() * 100
    f_in = entrenamiento.loc[~fuera & entrenamiento[col].notna(), OBJETIVO].mean() * 100
    print(f"{col:26s}{fuera.sum():>10d}{f_out:>15.2f}%{f_in:>15.2f}%{f_out/f_in:>8.2f}x")

Tasa de falla global en train: 8.88%

variable                    atípicos  falla en atíp.  falla normales    lift
----------------------------------------------------------------------------
operating_hours                   19          15.79%           8.84%    1.79x
temperature_c                     32          25.00%           8.74%    2.86x
vibration_mm_s                    17           0.00%           9.01%    0.00x
pressure_bar                      25          16.00%           8.64%    1.85x
days_since_maintenance            81          18.52%           8.67%    2.14x
error_count_30d                   37           8.11%           8.89%    0.91x
energy_kw                         30          10.00%           8.87%    1.13x


In [10]:
# ¿Cuánto costaría eliminar las filas atípicas?
mascara = np.zeros(len(entrenamiento), dtype=bool)
for col in NUMERICAS:
    lo, hi = limites[col]
    mascara |= ((entrenamiento[col] < lo) | (entrenamiento[col] > hi)).fillna(False).values

print(f"Filas con al menos un atípico : {mascara.sum()} ({mascara.mean()*100:.2f}% de train)")
print(f"  tasa de falla en esas filas : {entrenamiento.loc[mascara, OBJETIVO].mean()*100:.2f}%")
print(f"  tasa de falla en el resto   : {entrenamiento.loc[~mascara, OBJETIVO].mean()*100:.2f}%")
pct = entrenamiento.loc[mascara, OBJETIVO].sum() / entrenamiento[OBJETIVO].sum() * 100
print(f"\nEsas filas contienen el {pct:.2f}% de TODAS las fallas de train.")
print(f"Eliminarlas borraría {entrenamiento.loc[mascara, OBJETIVO].sum()} fallas de {entrenamiento[OBJETIVO].sum()}.")

Filas con al menos un atípico : 226 (6.03% de train)
  tasa de falla en esas filas : 14.60%
  tasa de falla en el resto   : 8.51%

Esas filas contienen el 9.91% de TODAS las fallas de train.
Eliminarlas borraría 33 fallas de 333.


**Decisión: se conservan TODOS los valores atípicos.** La evidencia, no la apariencia:

1. **Los atípicos fallan más, no menos.** En `temperature_c` la tasa de falla entre los atípicos es del 25.00 % contra 8.74 % en los normales — casi el triple. En `days_since_maintenance`, 18.52 % contra 8.67 %. En un problema de mantenimiento predictivo esto tiene sentido físico: una máquina que se sale de su rango normal de temperatura o que lleva 300 días sin mantenimiento **es exactamente la que está por fallar**. El atípico es la señal, no el ruido.

2. **Eliminarlos destruiría la clase minoritaria.** Las filas con algún atípico son el 6.03 % de train pero concentran el 9.91 % de todas las fallas. Borrarlas quitaría unas 33 fallas de 333, en un problema donde la clase positiva ya es solo el 8.88 %.

3. **No hay ningún valor físicamente imposible.** Se revisó en la sección 3: ningún negativo, ningún rango absurdo. No hay evidencia de error de medición que justifique borrar.

**La única excepción parcial:** `vibration_mm_s` tiene 17 atípicos con 0 % de fallas (lift 0.00x), que sí parecen ruido de sensor. Pero 17 filas sobre 3 750 es demasiado poco para distinguir una señal real del azar, y eliminarlas sería una decisión basada en una muestra minúscula. **Se conservan, y se deja anotado como limitación.**

Como los atípicos se conservan, la mediana y `StandardScaler` son elecciones coherentes: la mediana no se deja arrastrar por ellos al imputar.

## 7. Codificación y escalamiento

El criterio es el algoritmo que se va a usar: `LogisticRegression`.

- **Numéricas → `StandardScaler`.** La regresión logística optimiza por descenso de gradiente y penaliza los coeficientes; con variables en escalas tan distintas como `operating_hours` (cientos a miles) y `vibration_mm_s` (0 a 7), sin escalar el modelo no converge bien y la penalización castiga injustamente a las variables de escala pequeña. **Aquí el escalamiento no es opcional.**

- **Categóricas → `OneHotEncoder(handle_unknown="ignore")`.** Las tres son nominales: no hay orden entre `A`, `B` y `C`, ni entre `seco`, `humedo` y `polvoriento`. Codificarlas con números enteros le haría creer al modelo que C > B > A, que es falso. `handle_unknown="ignore"` evita que el pipeline se rompa si en producción aparece un ambiente que no estaba en train.

- **No se usa target encoding** bajo ninguna forma: usa la variable objetivo para construir predictoras y `CONTEXT.md` 2.2 lo prohíbe expresamente.

In [11]:
preprocesador = ColumnTransformer([
    ("numericas", Pipeline([
        ("imputador", SimpleImputer(strategy="median")),
        ("escalador", StandardScaler()),
    ]), NUMERICAS),
    ("categoricas", Pipeline([
        ("imputador", SimpleImputer(strategy="most_frequent")),
        ("codificador", OneHotEncoder(handle_unknown="ignore")),
    ]), CATEGORICAS),
])

preprocesador.fit(X_train)   # <- fit SOLO con train
print(f"Columnas antes  : {X_train.shape[1]}")
print(f"Columnas después: {preprocesador.transform(X_train).shape[1]}")
print(f"\nNombres generados:\n{list(preprocesador.get_feature_names_out())}")

Columnas antes  : 10
Columnas después: 16

Nombres generados:
['numericas__operating_hours', 'numericas__temperature_c', 'numericas__vibration_mm_s', 'numericas__pressure_bar', 'numericas__days_since_maintenance', 'numericas__error_count_30d', 'numericas__energy_kw', 'categoricas__machine_type_A', 'categoricas__machine_type_B', 'categoricas__machine_type_C', 'categoricas__shift_manana', 'categoricas__shift_noche', 'categoricas__shift_tarde', 'categoricas__environment_humedo', 'categoricas__environment_polvoriento', 'categoricas__environment_seco']


## 8. Selección de características

> **Nota de alcance.** `CONTEXT.md` v1.1 prohibía la selección de variables sin autorización escrita de la autora. La Tarea 2.2 la exige, así que la autorización quedó registrada y `CONTEXT.md` subió a v1.2. La regla no se saltó: se cambió a propósito y quedó documentado.

El criterio combina cuatro fuentes de evidencia, **todas medidas solo sobre train**.

In [12]:
print("=" * 72)
print("8.1  MULTICOLINEALIDAD — ¿hay variables que dicen lo mismo?")
print("=" * 72)
corr = X_train[NUMERICAS].corr().abs()
pares = sorted([(a, b, corr.loc[a, b]) for i, a in enumerate(NUMERICAS)
                for b in NUMERICAS[i+1:]], key=lambda t: -t[2])
for a, b, v in pares[:4]:
    print(f"  |r| = {v:.3f}   {a} ~ {b}")
print(f"\n  Máxima correlación: {pares[0][2]:.3f}. Muy por debajo de 0.8.")
print("  -> Ninguna variable se descarta por redundancia.")

8.1  MULTICOLINEALIDAD — ¿hay variables que dicen lo mismo?
  |r| = 0.222   operating_hours ~ vibration_mm_s
  |r| = 0.182   operating_hours ~ error_count_30d
  |r| = 0.167   vibration_mm_s ~ energy_kw
  |r| = 0.081   temperature_c ~ energy_kw

  Máxima correlación: 0.222. Muy por debajo de 0.8.
  -> Ninguna variable se descarta por redundancia.


In [13]:
print("=" * 72)
print("8.2  ASOCIACIÓN CON EL OBJETIVO")
print("=" * 72)
X_num = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X_train[NUMERICAS]),
                     columns=NUMERICAS)
mi = mutual_info_classif(X_num, y_train, random_state=SEMILLA)
print(f"{'variable':26s}{'corr. punto-biserial':>22s}{'información mutua':>20s}")
print("-" * 68)
for col, m in sorted(zip(NUMERICAS, mi), key=lambda t: -t[1]):
    r = np.corrcoef(X_num[col], y_train)[0, 1]
    print(f"{col:26s}{r:22.4f}{m:20.4f}")

print("\nCategóricas — tasa de falla por nivel:")
for col in CATEGORICAS:
    tabla = entrenamiento.groupby(col, dropna=False)[OBJETIVO].agg(["mean", "size"])
    detalle = ", ".join(f"{i}={v['mean']*100:.2f}% (n={int(v['size'])})"
                        for i, v in tabla.iterrows())
    print(f"  {col:14s} {detalle}")

8.2  ASOCIACIÓN CON EL OBJETIVO
variable                    corr. punto-biserial   información mutua
--------------------------------------------------------------------
vibration_mm_s                            0.1860              0.0332
operating_hours                           0.1009              0.0256
temperature_c                             0.0579              0.0237
energy_kw                                -0.0357              0.0167
pressure_bar                             -0.0219              0.0162
days_since_maintenance                    0.1244              0.0064
error_count_30d                           0.0697              0.0016

Categóricas — tasa de falla por nivel:
  machine_type   A=9.09% (n=1661), B=7.53% (n=1328), C=10.78% (n=761)
  shift          manana=7.78% (n=1363), noche=9.69% (n=939), tarde=9.39% (n=1448)
  environment    humedo=7.50% (n=987), polvoriento=9.75% (n=636), seco=9.26% (n=2041), nan=9.30% (n=86)


In [14]:
print("=" * 72)
print("8.3  ABLACIÓN — ¿quitar una variable mejora el modelo?")
print("=" * 72)
def pipeline_con(numericas, categoricas):
    return arma_pipeline(SimpleImputer(strategy="median"), numericas, categoricas)

base = cross_val_score(pipeline_con(NUMERICAS, CATEGORICAS), X_train, y_train,
                       cv=cv, scoring="f1").mean()
print(f"  Las 10 variables ............. F1 = {base:.4f}  (referencia)\n")
for col in NUMERICAS + CATEGORICAS:
    n2 = [c for c in NUMERICAS if c != col]
    c2 = [c for c in CATEGORICAS if c != col]
    s = cross_val_score(pipeline_con(n2, c2), X_train, y_train, cv=cv, scoring="f1").mean()
    marca = "  <-- quitarla parece mejorar" if s - base > 0.002 else ""
    print(f"  sin {col:24s} F1 = {s:.4f}  ({s-base:+.4f}){marca}")

8.3  ABLACIÓN — ¿quitar una variable mejora el modelo?
  Las 10 variables ............. F1 = 0.2650  (referencia)



  sin operating_hours          F1 = 0.2622  (-0.0028)


  sin temperature_c            F1 = 0.2714  (+0.0065)  <-- quitarla parece mejorar


  sin vibration_mm_s           F1 = 0.2324  (-0.0326)
  sin pressure_bar             F1 = 0.2690  (+0.0040)  <-- quitarla parece mejorar


  sin days_since_maintenance   F1 = 0.2556  (-0.0094)


  sin error_count_30d          F1 = 0.2686  (+0.0036)  <-- quitarla parece mejorar


  sin energy_kw                F1 = 0.2504  (-0.0146)
  sin machine_type             F1 = 0.2721  (+0.0071)  <-- quitarla parece mejorar


  sin shift                    F1 = 0.2747  (+0.0097)  <-- quitarla parece mejorar


  sin environment              F1 = 0.2626  (-0.0024)


### 8.4 ¿Esas mejoras son reales o son ruido?

Este es el paso que separa una selección seria de una casualidad. Cinco particiones no bastan para afirmar que una diferencia de +0.007 significa algo. Repito la validación cruzada 6 veces (30 mediciones) y aplico una **prueba t pareada**.

In [15]:
cv_repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=SEMILLA)
candidatos = {
    "10 variables (todas)":         (NUMERICAS, CATEGORICAS),
    "sin error_count_30d":          ([c for c in NUMERICAS if c != "error_count_30d"], CATEGORICAS),
    "sin shift":                    (NUMERICAS, [c for c in CATEGORICAS if c != "shift"]),
    "sin error_count_30d y shift":  ([c for c in NUMERICAS if c != "error_count_30d"],
                                     [c for c in CATEGORICAS if c != "shift"]),
}

puntajes = {}
for nombre, (nu, ca) in candidatos.items():
    s = cross_val_score(pipeline_con(nu, ca), X_train, y_train,
                        cv=cv_repetida, scoring="f1")
    puntajes[nombre] = s
    print(f"{nombre:30s} F1 = {s.mean():.4f}  ± {s.std():.4f}")

referencia = puntajes["10 variables (todas)"]
print("\nPrueba t pareada contra el modelo de 10 variables:")
for nombre, s in puntajes.items():
    if nombre == "10 variables (todas)":
        continue
    t, p = stats.ttest_rel(s, referencia)
    veredicto = "DIFERENCIA REAL" if p < 0.05 else "dentro del ruido"
    print(f"  {nombre:30s} Δ = {s.mean()-referencia.mean():+.4f}   p = {p:.4f}   -> {veredicto}")

10 variables (todas)           F1 = 0.2715  ± 0.0205


sin error_count_30d            F1 = 0.2739  ± 0.0210


sin shift                      F1 = 0.2760  ± 0.0208


sin error_count_30d y shift    F1 = 0.2732  ± 0.0186

Prueba t pareada contra el modelo de 10 variables:
  sin error_count_30d            Δ = +0.0024   p = 0.4268   -> dentro del ruido
  sin shift                      Δ = +0.0045   p = 0.0093   -> DIFERENCIA REAL
  sin error_count_30d y shift    Δ = +0.0017   p = 0.5601   -> dentro del ruido


### Decisión: se descarta `shift`. Se conservan las otras nueve.

**Se descarta `shift`** por tres razones que apuntan en la misma dirección:

- **Estadística.** Es el único cambio cuya mejora supera el ruido de medición: Δ = +0.0045 con p = 0.0093 sobre 30 mediciones pareadas. Todos los demás candidatos dan p > 0.4.
- **Empírica.** Sus tres niveles tienen tasas de falla casi idénticas (7.78 %, 9.69 %, 9.39 %): apenas distingue nada, pero cuesta dos columnas tras el one-hot.
- **De dominio.** El turno en que opera una máquina es una etiqueta administrativa, no una condición física. No hay un mecanismo por el cual el turno de la tarde haga fallar un rodamiento. Si apareciera una señal fuerte en el turno, lo más probable sería que estuviera capturando otra cosa (carga, operador, temperatura ambiente) y no el turno en sí.

**Se conserva `error_count_30d` aunque su información mutua sea la más baja (0.0016).** Quitarla da Δ = +0.0024 con p = 0.4268: está dentro del ruido. Y el conocimiento del dominio dice que el número de errores recientes es un indicador legítimo del estado de una máquina. **Sin evidencia para eliminarla, la decisión conservadora es conservarla.** Descartar una variable plausible por una mejora que no supera el azar sería sobreajustar la decisión a esta partición concreta.

Resultado: **9 variables** (7 numéricas + `machine_type` + `environment`).

## 9. Evaluación final en test

Primera y única vez que se toca `X_test`.

In [16]:
CATEGORICAS_FINALES = [c for c in CATEGORICAS if c != "shift"]

def evalua(nombre, numericas, categoricas):
    Xs = df[numericas + categoricas]
    Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=TEST_SIZE,
                                          random_state=SEMILLA, stratify=y)
    p = pipeline_con(numericas, categoricas).fit(Xtr, ytr)
    pred = p.predict(Xte)
    print(f"--- {nombre} ---")
    print(f"  accuracy     {accuracy_score(yte, pred):.4f}")
    print(f"  precisión(1) {precision_score(yte, pred):.4f}")
    print(f"  recall(1)    {recall_score(yte, pred):.4f}")
    print(f"  F1(1)        {f1_score(yte, pred):.4f}")
    mc = confusion_matrix(yte, pred)
    print(f"  matriz de confusión:\n{mc}")
    print(f"  fallas detectadas: {mc[1,1]} de {mc[1,0]+mc[1,1]}\n")

evalua("ANTES — Tarea 1.2, 10 variables", NUMERICAS, CATEGORICAS)
evalua("DESPUÉS — Tarea 2.2, 9 variables", NUMERICAS, CATEGORICAS_FINALES)

--- ANTES — Tarea 1.2, 10 variables ---
  accuracy     0.6856
  precisión(1) 0.1527
  recall(1)    0.5586
  F1(1)        0.2398
  matriz de confusión:
[[795 344]
 [ 49  62]]
  fallas detectadas: 62 de 111

--- DESPUÉS — Tarea 2.2, 9 variables ---
  accuracy     0.6872
  precisión(1) 0.1667
  recall(1)    0.6306
  F1(1)        0.2637
  matriz de confusión:
[[789 350]
 [ 41  70]]
  fallas detectadas: 70 de 111



**Lectura de los resultados.**

| Métrica | Antes (10 var.) | Después (9 var.) |
|---|---|---|
| Accuracy | 0.6856 | 0.6872 |
| Precisión (clase 1) | 0.1527 | 0.1667 |
| **Recall (clase 1)** | **0.5586** | **0.6306** |
| **F1 (clase 1)** | **0.2398** | **0.2637** |
| Fallas detectadas | 62 de 111 | **70 de 111** |

El modelo final detecta **8 fallas más** sobre las mismas 111 del conjunto de prueba, con una variable menos. En mantenimiento predictivo el recall es la métrica que importa: cada falla no detectada es una máquina que se rompe sin aviso, mientras que un falso positivo solo cuesta una inspección.

**Y hay que decir lo que el modelo todavía no hace bien:** con una precisión de 0.17, de cada 6 máquinas que el modelo marca, solo 1 falla de verdad. Sirve para *priorizar inspecciones* — que es el uso previsto declarado en `CONTEXT.md` — pero no para tomar decisiones automáticas.

## 10. Verificación de que no hay fuga de información

Estas pruebas están también en `verificar_fuga.py`, que puede ejecutarse desde la terminal. Aquí se corren dentro del cuaderno.

In [17]:
!python3 verificar_fuga.py

[PASA] 1. La variable objetivo no esta entre las predictoras
         predictoras = ['operating_hours', 'temperature_c', 'vibration_mm_s', 'pressure_bar', 'days_since_maintenance', 'error_count_30d', 'energy_kw', 'machine_type', 'environment']
[PASA] 2. El escalador aprendio de train, no del conjunto completo
         media train[0] = 5185.9965 vs media global[0] = 5187.1950
[PASA] 3. La imputacion aprendio la mediana de train, no la global
         medianas train = [5.1950e+03 6.9125e+01 3.8500e+00 6.3200e+00 8.1000e+01 2.0000e+00
 4.6320e+01]


[PASA] 4. Cada fila de test se transforma sin mirar a las demas
         transformar en bloque == transformar fila por fila
[PASA] 5. Ninguna fila aparece en train y en test a la vez
         filas compartidas = 0


[PASA] 6a. Fuga por imputacion/escalamiento: medida, no supuesta
         CON fuga F1 = 0.2637 | SIN fuga F1 = 0.2637 | diferencia = 0.0000
         Con n = 5000 las medianas de train y del conjunto completo
         difieren como maximo 0.259 %, asi que esta fuga concreta
         es INOCUA en este dataset. No se reporta como si hubiera
         inflado el resultado, porque no lo hizo.


[PASA] 6b. Fuga por seleccion de variables: esta SI infla el resultado
         seleccionando con TODO  -> F1 medio = 0.2808
         seleccionando con TRAIN -> F1 medio = 0.2746
         la fuga infla el F1 en +0.0062 sobre 20 muestras.
         Por eso la seleccion de variables de la seccion 5 se decidio
         con validacion cruzada SOBRE TRAIN y el test no se toco.

RESULTADO: 7/7 pruebas pasaron. El pipeline no tiene fuga de informacion.


### Lo que las pruebas encontraron, dicho con honestidad

La prueba 6a mide cuánto infla el resultado hacer la imputación y el escalamiento **antes** del split. La respuesta en este dataset es: **nada, 0.0000 de diferencia en F1.**

No lo voy a maquillar. La razón es concreta: con 5 000 filas y solo 2–3 % de faltantes, las medianas de train y las del conjunto completo difieren como máximo un 0.26 %. La fuga existe conceptualmente pero es numéricamente inocua aquí.

**Eso no significa que el orden dé igual.** Significa dos cosas:

1. El orden correcto es una **póliza de seguro**. Con menos filas, más faltantes, o un escalador sensible a atípicos, la diferencia sí aparece — y no se puede saber de antemano si un dataset es benigno sin haberlo medido.
2. **La fuga que sí importa en esta tarea es otra:** la de la selección de características. La prueba 6b la mide eligiendo variables con todo el conjunto contra elegirlas solo con train, y ahí sí hay inflación del F1. Es precisamente el paso nuevo que introduce la Tarea 2.2, y por eso toda la sección 8 se decidió con validación cruzada sobre train.

## 11. Conjunto de datos final, listo para entrenar

In [18]:
X_final = df[NUMERICAS + CATEGORICAS_FINALES]
Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(
    X_final, y, test_size=TEST_SIZE, random_state=SEMILLA, stratify=y)

pipeline_final = pipeline_con(NUMERICAS, CATEGORICAS_FINALES).fit(Xtr_f, ytr_f)
prep_final = pipeline_final.named_steps["preprocesamiento"]

print("CONJUNTO FINAL")
print(f"  filas totales ........... {len(df):,}")
print(f"  train / test ............ {len(Xtr_f):,} / {len(Xte_f):,}")
print(f"  variables de entrada .... {X_final.shape[1]}  (10 originales menos 'shift')")
print(f"  columnas tras codificar . {prep_final.transform(Xtr_f).shape[1]}")
print(f"  valores faltantes ....... {int(pd.DataFrame(prep_final.transform(Xtr_f)).isna().sum().sum())}")
print(f"  clase positiva .......... {y.mean()*100:.2f}%")
print(f"\n  columnas finales:\n  {list(prep_final.get_feature_names_out())}")

# Guardar el conjunto transformado como evidencia reproducible
os.makedirs("reportes", exist_ok=True)
salida = pd.DataFrame(prep_final.transform(Xtr_f),
                      columns=prep_final.get_feature_names_out())
salida[OBJETIVO] = ytr_f.values
salida.to_csv("reportes/train_preprocesado.csv", index=False)
print(f"\n  guardado: reportes/train_preprocesado.csv  {salida.shape}")

CONJUNTO FINAL
  filas totales ........... 5,000
  train / test ............ 3,750 / 1,250
  variables de entrada .... 9  (10 originales menos 'shift')
  columnas tras codificar . 13
  valores faltantes ....... 0
  clase positiva .......... 8.88%

  columnas finales:
  ['num__operating_hours', 'num__temperature_c', 'num__vibration_mm_s', 'num__pressure_bar', 'num__days_since_maintenance', 'num__error_count_30d', 'num__energy_kw', 'cat__machine_type_A', 'cat__machine_type_B', 'cat__machine_type_C', 'cat__environment_humedo', 'cat__environment_polvoriento', 'cat__environment_seco']



  guardado: reportes/train_preprocesado.csv  (3750, 14)


---

## Resumen de decisiones

| Paso | Decisión | Evidencia que la respalda |
|---|---|---|
| Faltantes | Imputar con la **mediana** | Las tres estrategias empatan (F1 0.2650 / 0.2647 / 0.2621); gana la más simple y la que no añade columnas |
| Duplicados | Quitarlos **antes** del split | No había ninguno, pero el código queda: una copia a ambos lados de la partición es fuga directa |
| Atípicos | **Conservarlos todos** | Fallan 2–3 veces más que las filas normales; contienen el 9.91 % de todas las fallas |
| Codificación | **One-hot** | Las tres categóricas son nominales, sin orden |
| Escalamiento | **StandardScaler** | Obligatorio para regresión logística por escalas muy dispares |
| Selección | Descartar **`shift`** | Único cambio con p < 0.05 (p = 0.0093); niveles casi idénticos; sin mecanismo físico |
| Selección | Conservar `error_count_30d` | Quitarla está dentro del ruido (p = 0.43) y el dominio la respalda |

**Autora:** Araceli Castillo · MAI 540 · Atlantis University · Septiembre de 2026